In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("landslide_dataset.csv")

# ── 5% Gaussian noise on continuous features ──────────────────────
NOISE_LEVEL = 0.05   # 5% of each column's std dev

continuous_cols = [
    'Rainfall_mm', 'Slope_Angle', 'Soil_Saturation',
    'Vegetation_Cover', 'Earthquake_Activity', 'Proximity_to_Water'
]

np.random.seed(42)
for col in continuous_cols:
    std = df[col].std()
    noise = np.random.normal(0, NOISE_LEVEL * std, size=len(df))
    df[col] = df[col] + noise

# ── 5% random label flips (simulates annotation error) ────────────
n_flip = int(0.05 * len(df))
flip_idx = np.random.choice(df.index, size=n_flip, replace=False)
df.loc[flip_idx, 'Landslide'] = 1 - df.loc[flip_idx, 'Landslide']

print(f"Noise added to {len(continuous_cols)} continuous columns")
print(f"Flipped {n_flip} labels ({NOISE_LEVEL*100:.0f}% of {len(df)} rows)")
print("\nClass balance after noise:")
print(df['Landslide'].value_counts())
df.head()

Noise added to 6 continuous columns
Flipped 100 labels (5% of 2000 rows)

Class balance after noise:
Landslide
1    1006
0     994
Name: count, dtype: int64


,Rainfall_mm,Slope_Angle,Soil_Saturation,Vegetation_Cover,Earthquake_Activity,Proximity_to_Water,Landslide,Soil_Type_Gravel,Soil_Type_Sand,Soil_Type_Silt
0,207.813337,57.762679,0.880032,0.324957,4.386916,0.080300,1,0,0,0
1,218.432965,36.570141,0.656685,0.352140,4.093284,0.823584,1,0,0,1
2,183.979323,30.762589,0.673628,0.209430,5.295550,0.005739,1,0,0,1
3,233.721232,38.761454,0.625653,0.482367,4.649683,0.788378,1,0,0,1
4,179.181584,41.561340,0.816759,0.115792,5.640000,0.485215,1,0,0,0


In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop("Landslide", axis=1)
Y = df["Landslide"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1600, 9) Test: (400, 9)


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\n", classification_report(y_test, y_pred))

Accuracy: 0.95
ROC-AUC: 0.951686292157304

               precision    recall  f1-score   support

           0       0.95      0.95      0.95       199
           1       0.95      0.95      0.95       201

    accuracy                           0.95       400
   macro avg       0.95      0.95      0.95       400
weighted avg       0.95      0.95      0.95       400



In [27]:
# Save both for fusion step
import joblib
joblib.dump(model,  'landslide_rf.pkl')
joblib.dump(scaler, 'landslide_scaler.pkl')
print("\nSaved: landslide_rf.pkl, landslide_scaler.pkl")


Saved: landslide_rf.pkl, landslide_scaler.pkl
